# 03 — Modeling and Evaluation


Goals:
- Establish a majority-class baseline
- Train candidate models
- Use consistent preprocessing and splits
- Compare classification metrics
- Add cross-validation and tuning


## Baseline Classification

Before comparing machine learning models, we establish a naive baseline.

Because the target is imbalanced, the baseline classifier predicts the
most frequent class for every observation.

Future models should be compared against this baseline rather than
considering accuracy in isolation.

## Load the data

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR
sys.path.append(str(PROJECT_ROOT))

from src.evaluation import (
    evaluate_binary_classifier,
    get_confusion_matrix,
    results_to_dataframe,
)

DATA_PATH = PROJECT_ROOT / "data" / "bank-full.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download bank-full.csv and place it in the data/ folder."
    )

df = pd.read_csv(
    DATA_PATH,
    sep=";",
)




## Temporary Train/Test Split

In [ ]:
df.head()

X = df.drop(columns=["y"])

y = df["y"].map({
    "no": 0,
    "yes": 1,
})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

## Create a temporary baseline model

In [ ]:
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_train,
    y_train,
)

baseline_predictions = baseline.predict(
    X_test
)



## Evaluate the baseline model


In [ ]:

baseline_results = evaluate_binary_classifier(
    model_name="Majority Class Baseline",
    y_true=y_test,
    y_pred=baseline_predictions,
)

baseline_results

## Comparison table

In [ ]:


results = [
    baseline_results
]

results_df = results_to_dataframe(results)

results_df

In [ ]:
baseline_cm = get_confusion_matrix(
    y_test,
    baseline_predictions,
)

baseline_cm